# Fase 2: Segmentacion CXR y CT

Este notebook entrena y evalua modelos de segmentacion binaria para mascaras pulmonares CXR y mascaras de infeccion CT. La ejecucion documentada para la memoria es siempre `full`.


## 1. Setup

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib_cache'))

import matplotlib.pyplot as plt
import pandas as pd
import torch

from src.config import config
from src.data.segmentation import (
    build_ct_segmentation_dataframe,
    build_cxr_segmentation_dataframe,
    split_segmentation_dataframe,
)
from src.training.segmentation_experiment import (
    SEGMENTATION_ARCHITECTURES,
    estimate_foreground_pos_weight,
    existing_segmentation_artifact,
    get_device,
    limit_segmentation_samples,
    make_segmentation_run_config,
    save_segmentation_artifacts,
    seed_everything,
    summarize_segmentation_results,
    train_and_evaluate_segmentation,
)

pd.set_option('display.max_columns', 50)
seed_everything(config.RANDOM_SEED)
device = get_device()
device

## 2. Parametros experimentales

Configuracion de experimentos completos. Los resultados de la memoria deben proceder de ejecuciones `full`.


In [ ]:
RUN_MODE = 'full'
RESUME_EXISTING = True

DATASETS = ['ct']
ARCHITECTURES = ['attention_unet']

# Hueco para experimentacion real de hiperparametros:
# - subir epochs si la validacion todavia mejora,
# - ajustar learning_rate si la validacion se estanca,
# - reducir batch_size si hay falta de memoria,
# - para CT, probar weighted_tversky_bce porque la lesion ocupa <1% de pixeles.
HYPERPARAMETER_OVERRIDES = {
    'ct': {
        'variant_name': 'tversky_pos30_bf16_thr',
        'loss_name': 'weighted_tversky_bce',
        'bce_weight': 0.2,
        'dice_weight': 0.8,
        'pos_weight': 30.0,
        'tversky_alpha': 0.3,
        'tversky_beta': 0.7,
        'optimize_threshold': True,
        'base_features': 16,
        'batch_size': 8,
        'epochs': 12,
        'early_stopping_patience': 4,
        'learning_rate': 1e-4,
    },
}

seg_models_dir = config.MODELS_DIR / 'segmentation'
seg_results_dir = PROJECT_ROOT / 'results' / 'segmentation'
processed_seg_dir = config.CT_DIR / 'processed_segmentation_slices'
seg_models_dir.mkdir(parents=True, exist_ok=True)
seg_results_dir.mkdir(parents=True, exist_ok=True)

print(f'RUN_MODE={RUN_MODE}')
print(f'RESUME_EXISTING={RESUME_EXISTING}')
print(f'DATASETS={DATASETS}')
print(f'ARCHITECTURES={ARCHITECTURES}')


## 3. Auditoria de pares imagen-mascara

In [ ]:
cxr_seg_df = build_cxr_segmentation_dataframe(config.CXR_DIR)
display(cxr_seg_df.groupby('label').size().rename('n_pares').reset_index())
print(f'CXR pares totales: {len(cxr_seg_df)}')

ct_seg_df = build_ct_segmentation_dataframe(
    ct_dir=config.CT_DIR,
    output_dir=processed_seg_dir,
    target_size=config.CT_IMAGE_SIZE,
    positive_mask_only=True,
    overwrite=False,
)
display(ct_seg_df.groupby('study_id').size().describe().to_frame('slices_con_mascara'))
print(f'CT slices con mascara positiva: {len(ct_seg_df)}')

## 4. Splits reproducibles

En CXR se estratifica por clase original. En CT se divide por `study_id` para evitar fuga de informacion entre slices del mismo estudio.

In [ ]:
splits = {}

cxr_train_df, cxr_val_df, cxr_test_df = split_segmentation_dataframe(
    cxr_seg_df,
    random_seed=config.RANDOM_SEED,
    stratify_col='label',
)
splits['cxr'] = (cxr_train_df, cxr_val_df, cxr_test_df)

ct_train_df, ct_val_df, ct_test_df = split_segmentation_dataframe(
    ct_seg_df,
    random_seed=config.RANDOM_SEED,
    group_col='study_id',
)
splits['ct'] = (ct_train_df, ct_val_df, ct_test_df)

split_summary = []
for dataset_name, (train_df, val_df, test_df) in splits.items():
    split_summary.append({'dataset': dataset_name, 'split': 'train', 'n': len(train_df), 'studies': train_df['study_id'].nunique()})
    split_summary.append({'dataset': dataset_name, 'split': 'val', 'n': len(val_df), 'studies': val_df['study_id'].nunique()})
    split_summary.append({'dataset': dataset_name, 'split': 'test', 'n': len(test_df), 'studies': test_df['study_id'].nunique()})
display(pd.DataFrame(split_summary))

## 5. Entrenamiento y evaluacion

In [ ]:
all_summaries = []
latest_result = None

for dataset_name in DATASETS:
    train_df, val_df, test_df = splits[dataset_name]
    train_df = limit_segmentation_samples(train_df, RUN_MODE)
    val_df = limit_segmentation_samples(val_df, RUN_MODE, max_samples=24)
    test_df = limit_segmentation_samples(test_df, RUN_MODE, max_samples=24)

    image_size = config.CXR_IMAGE_SIZE if dataset_name == 'cxr' else config.CT_IMAGE_SIZE
    in_channels = 1
    overrides = HYPERPARAMETER_OVERRIDES.get(dataset_name, {}).copy()
    if dataset_name == 'ct' and overrides.get('pos_weight') == 'auto':
        overrides['pos_weight'] = estimate_foreground_pos_weight(train_df, max_pos_weight=30.0)
        print(f"CT pos_weight estimado: {overrides['pos_weight']:.2f}")

    for architecture in ARCHITECTURES:
        run_config = make_segmentation_run_config(
            dataset_name=dataset_name,
            architecture=architecture,
            run_mode=RUN_MODE,
            image_size=image_size,
            in_channels=in_channels,
            **overrides,
        )
        print(f"\n=== {run_config.experiment_name}_{run_config.run_mode} ===")

        if RESUME_EXISTING and existing_segmentation_artifact(run_config, seg_models_dir / dataset_name, seg_results_dir / dataset_name):
            print('Saltado: artefactos existentes detectados.')
            continue

        latest_result = train_and_evaluate_segmentation(
            run_config=run_config,
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
            device=device,
        )
        saved_paths = save_segmentation_artifacts(
            run_config,
            latest_result,
            seg_models_dir / dataset_name,
            seg_results_dir / dataset_name,
        )
        all_summaries.append(saved_paths['summary'])

summary_paths = sorted(seg_results_dir.glob('*/*_summary.json'))
summary_df = summarize_segmentation_results(summary_paths)
display(summary_df.sort_values(['dataset', 'dice'], ascending=[True, False]))

## 6. Visualizacion cualitativa

Muestra ejemplos del ultimo entrenamiento ejecutado: imagen, mascara real y mascara predicha.

In [ ]:
if latest_result is not None:
    examples = latest_result['examples'][:4]
    fig, axes = plt.subplots(len(examples), 3, figsize=(9, 3 * len(examples)))
    if len(examples) == 1:
        axes = axes[None, :]
    for row_idx, example in enumerate(examples):
        image = example['image'].squeeze().numpy()
        mask = example['mask'].squeeze().numpy()
        pred = (example['prediction'].squeeze().numpy() >= 0.5).astype(float)
        axes[row_idx, 0].imshow(image, cmap='gray')
        axes[row_idx, 0].set_title('Imagen')
        axes[row_idx, 1].imshow(mask, cmap='gray')
        axes[row_idx, 1].set_title('Mascara real')
        axes[row_idx, 2].imshow(pred, cmap='gray')
        axes[row_idx, 2].set_title('Prediccion')
        for ax in axes[row_idx]:
            ax.axis('off')
    plt.tight_layout()
else:
    print('No hay entrenamiento nuevo en esta ejecucion. Si se saltaron artefactos existentes, revisa los summaries guardados.')